# BANC Adjusted Influence Caulculator
- This notebook runs the Drugowitsch Lab's Python implementation of the Connectome Influence Calculator (v2.0.0) for the BANC connectome (v888) and this code was adapted from their analysis pipeline (https://github.com/DrugowitschLab/ConnectomeInfluenceCalculator)
- This analysis: 
    - loads and runs the calculator between a given seed or population of seed neurons and a target or population of target neurons
    - allows for customizable calculator settings (syanpse threshold, silencing, etc.)
    - processes the output based on desired settings/format


### Part 1: Imports
- The Influence Calculator repository can be downloaded from the Drugowitsch Lab's GitHub (https://github.com/DrugowitschLab/ConnectomeInfluenceCalculator)
- Note: running the calculator requires PETSc and SLEPc libraries that can be difficult to install. We have had success setting up a virtual conda environment and installing these libraries through conda before instaling the Influence Calculator 

In [46]:
# Part 1: Imports

import numpy as np
import pandas as pd
from InfluenceCalculator import InfluenceCalculator

### Part 2: Calculator Settings

- EDGE_FILE and META_FILE -> refer to the BANC connectome's connectivity table and neuorn-level metadata files, respectively, that the calculator uses to calculate influence
    - Currently, we are using the v888 BANC connectome's v3 edgelist (banc_888_edgelist_simple_v3.feather) and neuron metadata file (banc_888_meta.feather) that were both downloaded from the Bates et al., 2026 Harvard Dataverse page (https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/7WTH1N)

- SEED_IDS and TARGET_IDS -> refers to the list of BANC neuron IDs that will be used for calculating influence
    - Currently set up so that neuron IDs can be manully entered into the notebook

- SYNAPSE_THRESHOLD -> refers to the minimum weight a synaptic connection in the BANC edgelist can have to be included in the analysis
    - connections weaker than this threshold are excluded from the edgelist
    - default for BANC connectome = 3 synapses 

- SYN_WEIGHT_MEASURE -> specifies how the weight of each edge is represented when passed through the Influence Calculator, options:
    - "count" -> the raw synaptic weight of each edge is retained 
    - "norm" -> the raw synaptic weight of each edge is normalized to the post-synaptic neuron's total synaptic input 
        - "norm" is typically used to make neurons with different raw synaptic weights comparable 

- LAMBDA_MAX -> controls the extent to which the seed influence signal is allowed to "reverb" along the different long/recurrent pathways in the network
    - default = 0.99 for the large BANC connectome
    - lowering this value will lead to quicker signal decay and thus emphasizes shorter pathways 

- C_SHIFT -> set the degree to which the log-transformed Raw influence value is shifted for Adjusted Influence conversion
    - Adjusted Influence is calculated from raw influence (the calculator's output) using a log-transformation, followed by a constant shift (C_SHIFT) to make the scale readable
    - default = 24.0 (Bates et al., 2026)

- SILENCE_NONSEED_SENSORY -> optionally prevents neurons annotated as "sensory" or "ascending_sensory" from propagating signal during the Influence Calculation 
    - When calculating influence using a sensory neuron or sensory population as the seed, this setting helps isolate that seed's independent ability to propagate signal to the target and not its ability to recruit other sensory pathways 
    - defulat for sensory->target calcualtions = True 

- MANUAL_SILENCE_IDSs -> beyond the default silencing of sensory neurons (see above), this toggle allows for the silencing of additional neurons 
    - silencing a given neuron prevents it from propagating any signal through the network
    - manually enter neuron IDs to this variable 
    - often used to assess the relative impact a given intermediate neuron(s) has on baseline circuit influence 

- RUN_MODE -> determines how we analyze influence scores for a population of seed neurons, options: 
    - "simultaneous" -> when there are multiple seed neurons, simultaneous mode will activate all seeds together so that the resulting influence reflects the combined activity of the full source population
        - when in "simultaneous" mode, the calculator's raw output is the combined score of all seeds, for each target neuron
        - for each target, combined output is then divided by the number of seed neurons
        - target neuron scores are averaged together to get a target population score
        - the raw target influence score is then converted into Adjusted Influence
    - "per_seed" -> when there are mutliple seed neurons, per-seed mode activates each seed individually/separately so that the resulting influence refelcts the activity of just one neuron
        - when in "per_seed" mode, the calculator outputs a series of target influence scores for each seed neuron
        - these scores are appended to an all-seeds x all-target influene score table that is exported 
        - "per_seed" mode takes longer as each seed is being propagated one after another

- PER_SEED_OUTPUT_CSV -> only applies when RUN_MODE is set to "per_seed"
    - specifies the name of the output .csv file that will contain the full seeds x targets influence scores table 

In [47]:
# Part 2: Calculator Settings

# Enter specific connectivity data/connectome file to analyze
EDGE_FILE = "data/banc_888_edgelist_simple_v3.feather"

# Enter the corresponding neuron-level metadata file
META_FILE = "data/banc_888_meta.feather"

# Specify which neuron(s) will be the seed(s) and target(s)
SEED_IDS = pd.read_csv('data/yeast_orns.csv')['root_id']

TARGET_IDS = pd.read_csv('data/ovidns.csv')['root_id']

SYNAPSE_THRESHOLD = 3 # Minimum synaptic weight for edges
SYN_WEIGHT_MEASURE = "norm" # "count" or "norm"
LAMBDA_MAX = 0.99 # network reverb, default = 0.99 for BANC
C_SHIFT = 24.0 # constant shift used for Adjusted Influence

# Silence all non-seed sensory and ascending-sensory neurons
SILENCE_NONSEED_SENSORY = True

# Optional additional neurons to silence
MANUAL_SILENCE_IDS = [
    # Enter BANC neuron IDs here for silencing, separate with commas

]

# Analysis mode: "simultaneous" or "per_seed"
RUN_MODE = "simultaneous"

# Output file used during "per_seed" analysis mode
PER_SEED_OUTPUT_CSV = "ovidn_ca_pop_raw.csv"

### Part 3: Run the Calculator

- Step 1: Helper functions 
    - run_influence(...) -> wrapper around the Drugoswitsch Lab's "ic.calculate_influence" function that handles our custom silencing settings 
    - extract_target_raw_scores(...) -> extracts the specific target scores from the calculator's raw influence output
    - adjusted_from_raw(...) -> applies the Bates et al., Adjusted Influence transformation from raw influence (natural log(raw influence)+24.0)

- Step 2: Prepare BANC Data 
    - load and clean neuron IDs for seeds, targets, and silening lists
    - prepare conenctivity and neuron metadata files 
        - only include neurons that are labeled as proofread or roughly proofread
        - filter edgelist to only contain edges between these filtered neurons 
        - identify which neurons are annotated as sensory or ascnending sensory 

- Step 3: Validate Neurons and Silencing List 
    - check if the provided seed, target, and silenced neuron IDs are in our filtered graph, stop the run if some are absent 
    - build final silencing list based on our silence_nonseed_sensory and manual silence settings in Part 2

- Step 4: Build Influence Calculator 
    - initialize the Drugoswitsch Lab's Influence Calculator using our custom settings
    - print the current settings for that run in the notebook

- Step 5: Run Influence Calculator and Process Output
    - If in simultaneous mode: 
        - run all seeds together for a combined influence score for each target neuron 
        - for each target, divide the combined influence by the number of seed neurons 
        - average across all targets to get population-level raw influence 
        - convert to Adjusted Influence
        - print both results in notebook 
    - If in per-seed mode: 
        - run each seed individually to get a single neuron's raw influence score for each target neuron
        - loop through all the seed neurons, getting scores for each target, append them to a combined dataframe
        - generate a final all seeds x all targets raw influence score table 
        - save this table to a csv and export for external analysis 
    

In [48]:
# Part 3: Run the Calculator 


# Step 1: Helper Functions 

# define function that creates a wrapper around the ic.calculate_influence(), using silencing settings
def run_influence(active_seed_ids):
    active_seed_ids = [str(x) for x in active_seed_ids] # convert the seed IDs to strings
    silenced_ids = set(BASELINE_SILENCED_IDS) # get list of neuron IDs being silenced 
    if silenced_ids: # if there are neurons in the silenced list, run them in the calculator (ic.calculate_inflluence)
        return ic.calculate_influence( 
            active_seed_ids, # select the active seed neurons 
            silenced_neurons=sorted(silenced_ids), # select the neurons to be silenced 
        )
    return ic.calculate_influence(active_seed_ids) # if no neurons are silenced, run the calculator with only the active seeds 


# define function that extracts raw Influence scores for the target neurons
def extract_target_raw_scores(influence_df): # searches in the calculator's output result file
    scores = (
        influence_df[
            ["id", "Influence_score_(unsigned)"] # keep the neuron IDs and their raw influence scores 
        ]
        .copy() # create a copy ('socres') of this selected data
    )
    scores["id"] = scores["id"].astype(str) # convert neuron IDs to strings 
    raw_by_id = (
        scores
        .set_index("id")["Influence_score_(unsigned)"] # organize the raw influence scores by neuorn ID
    )
    return pd.DataFrame({ # output a table containing each target neurons and their raw influence scores
        "target_id": TARGET_IDS, # list specific target neurons IDs for one column
        "raw_influence": [
            float(raw_by_id.loc[target_id]) # for a given target neuron, pull out the raw influence score 
            for target_id in TARGET_IDS # repeat for each target neuron
        ],
    }) # create dataframe of target neuron scores 


# define function that converts raw Influence into Adjusted Influence                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       o Adjusted Influence using C_SHIFT
def adjusted_from_raw(raw_value, c=C_SHIFT): # function takes a raw influence value and the C_SHIFT setting?
    raw_value = float(raw_value) # re-format raw value
    if raw_value <= 0:
        return 0.0 # if the raw value is 0, return 0 immediately 
    return max(0.0, np.log(raw_value) + c) # Apply transformation (natural log + constant shift)



# Step 2: Prepare BANC Data

SEED_IDS = [str(x) for x in SEED_IDS] # convert seed IDs to strings 
TARGET_IDS = [str(x) for x in TARGET_IDS] # convert target IDs to strings
MANUAL_SILENCE_IDS = [str(x) for x in MANUAL_SILENCE_IDS] # convert manual silence IDs to strings

edges = pd.read_feather(EDGE_FILE) # load BANC connectivity data
meta = pd.read_feather(META_FILE) # load BANC neuron metadata 

edges = edges[["pre", "post", "count", "norm"]].copy() # keep relevant connectivity columns 
edges["pre"] = edges["pre"].astype(str) # convert presynaptic neuron IDs to strings 
edges["post"] = edges["post"].astype(str) # convert postsynatpic neuron IDs to strings 

meta = meta.copy() # copy the metadata file before re-formatting 
meta["banc_888_id"] = meta["banc_888_id"].astype(str).str.strip() # clean BANC neuron IDs
meta["super_class"] = meta["super_class"].astype("string").str.strip().str.lower() # clean super-class annotations

proofread_mask = ( # identify neurons annotated as proofread or roughtly-proofread
    meta["proofread"].astype(str).str.upper().eq("TRUE") # mark neurons as valid if they are proofread OR
    | meta["roughly_proofread"].astype(str).str.upper().eq("TRUE") # mark neurs as valid if they are roughly_proofread
)

proofread_ids = set(meta.loc[proofread_mask, "banc_888_id"]) # grab the IDs of the neurons marked as valid proofread

edges = edges.loc[ # filter edges using proofread neurons 
    edges["pre"].isin(proofread_ids) # make sure presynaptic neuron is proofread
    & edges["post"].isin(proofread_ids) # make sure postsynaptic neuron is proofread 
].copy() # create a filtered copy of the edges table

all_graph_ids = pd.unique( # create list of all unique neuron IDs remaining after filtering 
    pd.concat([edges["pre"], edges["post"]], ignore_index=True) # add presynaptic and postsynaptic IDs to same list
).astype(str) # clean list of IDs 

graph_id_set = set(all_graph_ids) # create a set of graph IDs to check

meta_ic = pd.DataFrame({"root_id": all_graph_ids}) # finalize metadata table used by the calculator 

super_class_lookup = ( # search meta file for each valid neuron's super-class annotation 
    meta[["banc_888_id", "super_class"]] # filter meta table to just the neuron IDs and the super-class columns
    .drop_duplicates("banc_888_id") # make sure each BANC ID shows up only once 
    .rename(columns={"banc_888_id": "root_id"}) # rename columns to match meta_ic
)

# annotate each neuron ID in meta_ic with its corresponding super-class annotation in 'super_class_lookup'
meta_ic = meta_ic.merge(super_class_lookup, on="root_id", how="left") 



# Step 3: Validate Neurons and Define Silencing

missing_seeds = [ # identify seed neurons absent from filtered graph
    neuron_id for neuron_id in SEED_IDS # search for neuron IDs in the current Seed list
    if neuron_id not in graph_id_set # check if each neuron ID is in the set of valid graph IDs we made in step 2
]

missing_targets = [ # identify target neurons absent from filtered graph 
    neuron_id for neuron_id in TARGET_IDS # search for neruon IDs in the current Target list 
    if neuron_id not in graph_id_set # check if each neuron ID is in the set of valid graph IDs we made in step 2 
]

if missing_seeds: # stop the calculator if any seed IDs are not in the filtered BANC graph
    pd.Series(missing_seeds).to_csv('missing_seeds.csv', index=False)
    if len(missing_seeds) == len(SEED_IDS):
        raise ValueError(f"All seeds are absent from the proofread-filtered BANC graph.")
    
    print(f"{len(missing_seeds)} seeds missing. Continuing with analysis.")
    SEED_IDS = list(set(SEED_IDS) - set(missing_seeds)) # remove the missing IDs from the seed list

if missing_targets: # stop the calculator if any target IDs are not in the filtered BANC graph
    pd.Series(missing_targets).to_csv('missing_targets.csv', index=False)
    if len(missing_targets) == len(TARGET_IDS):
        raise ValueError(f"All targets are absent from the proofread-filtered BANC graph.")
    
    print(f"{len(missing_targets)} targets missing. Continuing with analysis.")
    TARGET_IDS = list(set(TARGET_IDS) - set(missing_targets)) # remove the missing IDs from the target list

missing_manual = [ # identify manual silenced neurons from filtered graph 
    neuron_id for neuron_id in MANUAL_SILENCE_IDS # search for neuron IDs in the current silenced list 
    if neuron_id not in graph_id_set # check if each neuron ID is in the set of valide graph IDs we made in step 2
]

if missing_manual: # run warning message if there are missing silenced neurons 
    print( # display warning message
        "Warning: manually silenced IDs are absent "
        "from filtered graph and will be ignored:"
    )
    for neuron_id in missing_manual: # pring each missing neuron ID
        print(neuron_id)
    MANUAL_SILENCE_IDS = [ # remove the missing IDs from the manual silence list 
        neuron_id for neuron_id in MANUAL_SILENCE_IDS
        if neuron_id in graph_id_set
    ]

baseline_silenced = set(MANUAL_SILENCE_IDS) # start building the final silence list with the manual silenced list

if SILENCE_NONSEED_SENSORY: # check if SILENCE_NONSEED_SENSORY setting is turned on 
    sensory_ids = set( # identify sensory IDs
        meta_ic.loc[ # search in current meta table 
            meta_ic["super_class"].isin(["sensory", "ascending_sensory"]), # look in the super-class column for any sensory or ascending sensory neurons 
            "root_id", # grab their corresponding neuron IDs 
        ].astype(str) # clean list of sensory IDs
    )
    sensory_ids.difference_update(SEED_IDS) # remove the run's seed neurons from the sensory list 
    baseline_silenced.update(sensory_ids) # add the remaining sensory neurons to the silenced list 

BASELINE_SILENCED_IDS = sorted(baseline_silenced) # save the final list of neurons for silencing 



# Step 4: Build Influence Calculator

ic = InfluenceCalculator( # Initialize the Drugoswitsch Lab's Influence Calculator using the settings in PArt 2
    edges, # use the proofread-filtered BANC edgelist 
    meta_ic, # us the filtered BANC neuron metadata
    signed=False, # keep calculator in unsigned mode (this current notebook cannot calculate signed influence)
    count_thresh=SYNAPSE_THRESHOLD, # apply the minimum synapse threshold 
    syn_weight_measure=SYN_WEIGHT_MEASURE, # used the syn weight meeasure setting (Part 2)
    lambda_max=LAMBDA_MAX, # use the lambda max setting (Part 2)
)

# Print and confirm calculator settings in notebook during run
print("\nBANC Adjusted Influence Analysis")
print(f"Run mode:                 {RUN_MODE}")
print(f"Synapse threshold:        {SYNAPSE_THRESHOLD}")
print(f"Synaptic weight:          {SYN_WEIGHT_MEASURE}")
print(f"Maximum eigenvalue:       {LAMBDA_MAX}")
print(f"Adjusted Influence shift: {C_SHIFT}")
print(f"Number of seeds:          {len(SEED_IDS)}") # count number of seed neurons 
print(f"Number of targets:        {len(TARGET_IDS)}") # count number of target neurons 
print(f"Silence non-seed sensory: {SILENCE_NONSEED_SENSORY}")
print(f"Manual silence neurons:   {len(MANUAL_SILENCE_IDS)}") # count number of manual silenced neurons



# Step 5: Run Influence Analysis

if RUN_MODE == "simultaneous": # run all seeds simultaneously for population-level influence calculation
    influence_df = run_influence(SEED_IDS) # run the influence calculator using the full seed population and our helper
    target_scores = extract_target_raw_scores(influence_df) # extract the combined influence of each target neuron using our helper

    target_scores["raw_influence_per_seed"] = ( # create a new column containing the per-seed influence
        target_scores["raw_influence"] / len(SEED_IDS) # to get per-seed influence, divide raw influence by the number of seeds 
    )

    raw_population_influence = float( # calculate population influence across all targets
        target_scores["raw_influence_per_seed"].mean() # average across the raw per-seed influence scores
    )

    adjusted_population_influence = adjusted_from_raw( # calculate adjusted influence using our helper
        raw_population_influence
    )

    print("\nPopulation-Level Influence") # format printed results table 
    print(f"Raw Influence:      {raw_population_influence:.16f}") # print the raw population-level influence
    print(f"Adjusted Influence: {adjusted_population_influence:.8f}") # print the population-level Adjusted Influence

elif RUN_MODE == "per_seed": # run each seed individually for neuron-level influence scores 
    per_seed_rows = [] # empty list to collect each seed's scores 

    for i, seed_id in enumerate(SEED_IDS, start=1): # loop through the seed neurons one after another 
        seed_influence = run_influence([seed_id]) # run the calculator using only the current seeds, with our helper
        seed_targets = extract_target_raw_scores(seed_influence) # extract the raw influence for the specified targets 
        seed_targets.insert(0, "seed_id", seed_id) # add the current neuron ID to that seed's results 
        per_seed_rows.append(seed_targets) # append the current seed's results to the combined list 

    per_seed_long = pd.concat(per_seed_rows, ignore_index=True) # combine the results from each seed's run 

    per_seed_results = per_seed_long.pivot( # reorgnaize the result table (all seeds x all targets)
        index="seed_id", # make each row represent a single seed neuron ID
        columns="target_id", # each column is a different target neuron 
        values="raw_influence", # the values are the raw_influence scores 
    )

    per_seed_results = per_seed_results.reindex(columns=TARGET_IDS) # maintain the same seed ID order as our input in Part 2

    per_seed_results.columns = [ # change each column title with its specific neuron ID
        f"raw influence to target {target_id}"
        for target_id in per_seed_results.columns
    ]

    per_seed_results = per_seed_results.reset_index() # reset the seed IDs index as a real column

    per_seed_results.to_csv( # save the all seeds x all target raw influence table as a csv 
        PER_SEED_OUTPUT_CSV, # use the file name specified in Part 2
        index=False,
    )

    print(f"Seeds analyzed: {len(per_seed_results):,}") # print the nubmer of seeds analyzed 
    print(f"Results saved to: {PER_SEED_OUTPUT_CSV}") # print the exported file name 

27 seeds missing. Continuing with analysis.

BANC Adjusted Influence Analysis
Run mode:                 simultaneous
Synapse threshold:        3
Synaptic weight:          norm
Maximum eigenvalue:       0.99
Adjusted Influence shift: 24.0
Number of seeds:          535
Number of targets:        16
Silence non-seed sensory: True
Manual silence neurons:   0

Population-Level Influence
Raw Influence:      0.0000072365207070
Adjusted Influence: 12.16362997
